In [ ]:
import xarray as xr
import os

SEASONAL_DIR = "/content/drive/MyDrive/Research Works - Development/Work 2 - SPI SPEI/Data/raster/Data for Dev/seasonal_aggregated"
YIELD_DIR    = "/content/drive/MyDrive/Research Works - Development/Work 2 - SPI SPEI/Data/raster/rounded_farm"
OUTPUT_DIR   = "/content/drive/MyDrive/Research Works - Development/Work 2 - SPI SPEI/Data/raster/Data for Dev/final_with_yield"

os.makedirs(OUTPUT_DIR, exist_ok=True)

YEARS = range(1991, 2024)


In [ ]:
for year in YEARS:
    print(f"🔄 Merging yield for {year}")

    # -------------------------------------------------
    # Load seasonal features
    # -------------------------------------------------
    seasonal_ds = xr.open_dataset(
        os.path.join(SEASONAL_DIR, f"seasonal_features_{year}.nc")
    )

    # -------------------------------------------------
    # Load yield
    # -------------------------------------------------
    yield_ds = xr.open_dataset(
        os.path.join(YIELD_DIR, f"wheat_farm_{year}.nc")
    )

    # -------------------------------------------------
    # Identify yield variable safely
    # -------------------------------------------------
    if "H_wheat_dot_hat" in yield_ds.data_vars:
        yvar = "H_wheat_dot_hat"
    elif "wheat_yield" in yield_ds.data_vars:
        yvar = "wheat_yield"
    else:
        raise ValueError(f"No yield variable found in {year}")

    # -------------------------------------------------
    # Ensure yield has no time dimension
    # (annual → spatial only)
    # -------------------------------------------------
    if "time" in yield_ds[yvar].dims:
        yield_da = yield_ds[yvar].squeeze("time", drop=True)
    else:
        yield_da = yield_ds[yvar]

    # -------------------------------------------------
    # Merge seasonal + yield
    # -------------------------------------------------
    final_ds = xr.merge(
        [seasonal_ds, yield_da.rename("yield")],
        compat="no_conflicts"
    )

    # -------------------------------------------------
    # Metadata
    # -------------------------------------------------
    final_ds.attrs.update({
        "title": "Seasonal Climate, Drought Indices, and Wheat Yield",
        "year": year,
        "season": "May–October",
        "yield_type": "Annual wheat yield",
        "created_by": "xarray seasonal-yield merge"
    })

    # -------------------------------------------------
    # Save
    # -------------------------------------------------
    out_file = os.path.join(OUTPUT_DIR, f"final_dataset_{year}.nc")
    final_ds.to_netcdf(out_file)

    print(f"✅ Saved {out_file}")

print("🎉 Yield merged successfully for all years!")


In [ ]:
ds = xr.open_dataset("/content/drive/MyDrive/Research Works - Development/Work 2 - SPI SPEI/Data/raster/Data for Dev/final_with_yield/final_dataset_1991.nc")
print(ds)